# Neural Form-Finding Prototype Pipeline

This notebook demonstrates the unified centroidal form-finding pipeline:
1. **Tessellation Building** (Numpy)
2. **Centroidal State Export** (JAX ready)
3. **Stage 0**: Initial Mapping
4. **Stage 1**: Geometric Validity Optimization
5. **Stage 2**: Static Physics Solver

In [3]:
import os
import sys

# 1. FORCE CPU (Must be done BEFORE importing jax)
os.environ["JAX_PLATFORM_NAME"] = "cpu"

# 2. SETUP PATHS
project_root = os.path.abspath(os.path.join('..'))
src_path = os.path.join(project_root, 'src')

if project_root not in sys.path:
    sys.path.append(project_root)
if src_path not in sys.path:
    sys.path.append(src_path)

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import copy

# Configure JAX precision
jax.config.update("jax_enable_x64", True)

from src.problem.config import load_config
from src.topology.builder import build_tessellation
from src.problem.conditions import apply_boundary_conditions, apply_loads, set_material_properties
from src.jax_backend.centroidal.state import CentroidalState
from src.jax_backend.centroidal.geometry import reconstruct_vertices
from src.jax_backend.centroidal.pipeline import forward_pipeline
from src.utils.visualization import plot_tessellation
from src.jax_backend.physics_solver.kinematics import rotation_matrix

print("Setup complete.")
print("JAX Platform:", jax.lib.xla_bridge.get_backend().platform)

XlaRuntimeError: UNIMPLEMENTED: default_memory_space is not supported.

## 1. Load Configuration & Build Tessellation
The configuration is loaded from YAML, which automatically fetches the patterns from `data/library/patterns.yaml`.

In [ ]:
# Path relative to the notebook (notebooks/ subfolder)
config_path = "../data/configs/turn3.yaml"
config = load_config(config_path)

print(f"Pattern loaded: {type(config.pattern).__name__}")
print(f"Grid size: {config.width}x{config.height}")

tessellation = build_tessellation(config.pattern, config.width, config.height)
set_material_properties(tessellation, config)
apply_boundary_conditions(tessellation, config)
apply_loads(tessellation, config)

print(f"Tessellation built with {len(tessellation.faces)} faces.")

## 2. Export to Centroidal State
We convert the Numpy-based topology to a JAX-compatible `CentroidalState`.

In [ ]:
cs_dict = tessellation.to_centroidal_state()
initial_state = CentroidalState(**{k: jnp.array(v) for k, v in cs_dict.items()})

print("Centroidal State exported.")

## 3. Run Pipeline
This executes Stage 0, 1, and 2 in sequence.

In [ ]:
target_params = {
    'type': config.target_type,
    'center': list(config.target_center),
    'radius': config.target_radius,
}

geom_weights = {
    'connectivity': config.w_connectivity,
    'non_intersection': config.w_non_intersection,
    'target': config.w_target,
    'arm_symmetry': config.w_arm_symmetry,
    'void_length': config.w_void_length,
    'void_collinear': config.w_void_collinear,
}

result = forward_pipeline(
    initial_state=initial_state,
    target_params=target_params,
    map_type=config.map_type,
    scale_factor=config.scale_factor,
    geom_weights=geom_weights,
    use_contact=config.use_contact,
    k_contact=config.k_contact,
    min_angle=config.min_angle * jnp.pi / 180.0,
    cutoff_angle=config.cutoff_angle * jnp.pi / 180.0,
    linearized_strains=config.linearized_strains,
)

print("Pipeline execution complete.")

## 4. Visual Comparison
Let's compare the three stages.

In [ ]:
def get_tess_at_state(state):
    c = state.face_centroids
    s = state.centroid_node_vectors
    verts_rec = reconstruct_vertices(c, s)
    
    tess_copy = copy.deepcopy(tessellation)
    new_verts = np.zeros_like(tess_copy.vertices)
    for i, face in enumerate(tess_copy.faces):
        for j, v_idx in enumerate(face.vertex_indices):
            new_verts[v_idx] = verts_rec[i, j]
    tess_copy.update_vertices(new_verts)
    return tess_copy

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Stage 0
tess0 = get_tess_at_state(result['mapped_state'])
plot_tessellation(tess0, ax=axes[0], title="Stage 0: Mapping", show_target=True, target_params=target_params)

# Stage 1
tess1 = get_tess_at_state(result['valid_state'])
plot_tessellation(tess1, ax=axes[1], title="Stage 1: Validated", show_target=True, target_params=target_params)

# Stage 2
sol = result['solution']
valid_state = result['valid_state']
c_eq = valid_state.face_centroids + sol.fields[:, :2]
R = jax.vmap(rotation_matrix)(sol.fields[:, 2])
s_eq = jnp.einsum('nij, nkj -> nki', R, valid_state.centroid_node_vectors)
equilibrium_state = valid_state._replace(face_centroids=c_eq, centroid_node_vectors=s_eq)

tess2 = get_tess_at_state(equilibrium_state)
plot_tessellation(tess2, ax=axes[2], title="Stage 2: Equilibrium", show_target=True, target_params=target_params)

plt.tight_layout()
plt.show()